
Integrantes do Grupo:
- Anselmo Faria Alvarez Júnior
- Augusto José Mangini dos Santos
- Julio Cesar da Silva


# Solução

In [103]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configuração do ambiente

In [104]:
!pip install pyspark

In [106]:
from pyspark.sql import SparkSession
from pyspark.sql import Row

from datetime import datetime

appName = 'Big Data'
master = 'local[*]'

spark = SparkSession.builder     \
    .master(master) \
    .appName(appName) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

## Leitura de dados

In [5]:
# Usar esta entrada para testes
# input_data = spark.sparkContext.textFile('file:///content/drive/My Drive/mcu/mcu_subset.csv')

In [107]:
# Usar esta entrada para entrega final
input_data = spark.sparkContext.textFile('file:///content/drive/My Drive/mcu/mcu.csv')

In [108]:
input_data.take(10)

[';character;line;movie;year;words;Adam McKay;Anna Boden;Art Marcum;Ashley Edward Miller;Chris McKenna;Christopher Ford;Christopher Markus;Christopher Yost;Craig Kyle;Don Payne;Drew Pearce;Edgar Wright;Eric Pearson;Erik Sommers;Geneva Robertson-Dworet;Hawk Ostby;James Gunn;Joe Cornish;Joe Robert Cole;John Francis Daley;Jon Watts;Jonathan Goldstein;Joss Whedon;Justin Theroux;Mark Fergus;Matt Holloway;Paul Rudd;Ryan Coogler;Ryan Fleck;Shane Black;Stephen McFeely;Zack Stentz',
 '0;TONY STARK;Oh, I get it.  You guys aren’t allowed to talk.  Is that it?  Are you not allowed to talk?;Iron Man;2008;22;False;False;True;False;False;False;False;False;False;False;False;False;False;False;False;True;False;False;False;False;False;False;False;False;True;True;False;False;False;False;False;False',
 '1;IRON MAN JIMMY;No.  We’re allowed to talk.;Iron Man;2008;6;False;False;True;False;False;False;False;False;False;False;False;False;False;False;False;True;False;False;False;False;False;False;False;False;Tru

## Exemplo de uso do pipeline

In [109]:
from transformers import pipeline

# Baixar e configurar pipeline do modelo
sentiment = pipeline('sentiment-analysis')



[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [110]:

result = sentiment("I am Iron Man")


In [111]:
result

[{'label': 'POSITIVE', 'score': 0.999076247215271}]

In [112]:
result[0]['label']

'POSITIVE'

## Solução

In [113]:
# Inclua outros personagens de sua escolha
# Novos nomes: 'loki', 'thor', 'natasha romanoff', 'maria hill', 'wanda maximoff', 'jarvis'
characters = {'tony stark', 'steve rogers', 'thanos', 'bruce banner', 'loki', 'thor', 'natasha romanoff', 'maria hill', 'wanda maximoff', 'jarvis'}

In [114]:
import re

# Modifique a solução para implementar a função Map
def line_sentiment(line) :
  campos = line.split(';')

  # Ignorar linhas onde nao seja possivel extrair falar
  if len(campos) < 3:
    return

  # normalizar maiscula e minuscula
  personagem = campos[1].strip().lower()
  fala = campos[2].strip()

  # Filtra personagens fola da analise e falas de multiplos personagens
  if personagem not in characters:
    return

  if fala == '':
    return

 # Classificacao do sentiment
  resultado = sentiment(fala)
  label = resultado[0]['label']
  polaridade = 1 if label == 'POSITIVE' else -1

  yield (personagem, (polaridade, 1))

In [115]:
s = input_data.flatMap(line_sentiment)

In [116]:
s.take(10)

[('tony stark', (-1, 1)),
 ('tony stark', (1, 1)),
 ('tony stark', (1, 1)),
 ('tony stark', (1, 1)),
 ('tony stark', (-1, 1)),
 ('tony stark', (1, 1)),
 ('tony stark', (-1, 1)),
 ('tony stark', (1, 1)),
 ('tony stark', (1, 1)),
 ('tony stark', (-1, 1))]

In [117]:
s.count()

5263

In [120]:
# Implemente e aplique um método reduce para acumulação dos sentimentos dos personagens
def sentimentTotal(acc, v):
  return (v[0]+acc[0], acc[1] + v[1])

s_total = s.reduceByKey(sentimentTotal)


In [121]:
sentiment("I am Iron Man.")


[{'label': 'POSITIVE', 'score': 0.9992565512657166}]

In [122]:
# Implemente e aplique um método para calculo do sentimento médio
def sentimentAvg(x):
  return x[0] / x[1]

s_avg = s_total.mapValues(sentimentAvg)

resultado_final = s_avg.collect()


# Resultado Final


Apresente o resultado final da sua análise completa.

In [123]:
for personagem, media in sorted(s_avg.collect()):
  print(f'{personagem}: {media:.2f}')

s_avg.coalesce(1).saveAsTextFile('/content/drive/My Drive/mcu/resultado_final')

bruce banner: -0.30
jarvis: -0.21
loki: -0.03
maria hill: -0.17
natasha romanoff: -0.13
steve rogers: -0.15
thanos: -0.10
thor: -0.03
tony stark: -0.09
wanda maximoff: -0.10
